# Mestres do XGH — Aula 10
## pandas: parar de escrever cem linhas

**Shift + Enter** roda a célula.

Última pílula separada do curso. Da Aula 11 em diante, o curso inteiro já é IA.

---
# 0. Criando os arquivos da aula

Rode esta célula primeiro. Ela gera os CSVs que a gente vai usar — inclusive um bem sujo,
de propósito.

In [ ]:
import csv, random, itertools

random.seed(5)
cidades = ["Santos", "Guaruja", "Praia Grande", "Sao Vicente"]
produtos = ["teclado", "mouse", "monitor", "headset", "webcam"]
precos = {"teclado": 150, "mouse": 80, "monitor": 900, "headset": 250, "webcam": 180}

linhas = []
for i in range(200):
    c = random.choice(cidades)
    p = random.choice(produtos)
    q = random.randint(1, 5)
    v = precos[p] * q * random.uniform(.9, 1.1)
    mes = random.randint(1, 6)
    linhas.append([f"2026-{mes:02d}-{random.randint(1, 28):02d}", c, p, q, round(v, 2)])

with open("vendas.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["data", "cidade", "produto", "quantidade", "valor"])
    w.writerows(linhas)

print("vendas.csv criado com", len(linhas), "linhas")

---
# 1. A pergunta que ficou de sábado passado

*"E se o CSV tivesse 5.000 linhas? Quantas linhas de código isso vai ter?"*

Vamos responder com o `vendas.csv` — 200 linhas, cinco colunas. Quero o **total por cidade**.

Do jeito que vocês já sabem fazer:

In [ ]:
import csv

total_por_cidade = {}

with open("vendas.csv", encoding="utf-8") as f:
    for linha in csv.DictReader(f):
        cidade = linha["cidade"]
        total_por_cidade[cidade] = total_por_cidade.get(cidade, 0) + float(linha["valor"])

for cidade in sorted(total_por_cidade):
    print(f"{cidade:14s} {total_por_cidade[cidade]:10.2f}")

Funciona perfeitamente. Usa `with`, `DictReader`, `for`, `.get()` com padrão, conversão de tipo,
f-string — **seis coisas que vocês aprenderam em nove aulas**, todas certas.

Agora o mesmo resultado:

In [ ]:
import pandas as pd

df = pd.read_csv("vendas.csv")
print(df.groupby("cidade")["valor"].sum().round(2))

```
cidade
Guaruja         52223.12
Praia Grande    56859.49
Santos          44765.02
Sao Vicente     50353.80
```

**Mesmos números. Duas linhas.**

> E aqui vai o aviso mais importante da aula: **não jogue fora as nove aulas anteriores.**
>
> O pandas é rápido de escrever porque alguém escreveu aquele `for` por você. Quem não sabe o que
> acontece por dentro fica preso no primeiro caso em que a biblioteca não faz exatamente o que
> precisa — e isso acontece toda semana.

---
# 2. DataFrame: a tabela que o Python entende

`read_csv` devolve um **DataFrame** — uma tabela com linhas, colunas e nomes.

In [ ]:
df = pd.read_csv("vendas.csv")

print(df.head())        # as 5 primeiras

In [ ]:
print(df.shape)         # (linhas, colunas)
print(list(df.columns))

Repare que o DataFrame junta duas coisas que vocês já conhecem: cada **coluna tem nome** (como um
dicionário) e cada **linha tem posição** (como uma lista). É a lista de dicionários da Aula 7,
mas com músculo.

`.describe()` é o primeiro comando que se roda em qualquer tabela nova — ele conta a história
inteira em cinco segundos:

In [ ]:
print(df[["quantidade", "valor"]].describe().round(2))

```
       quantidade    valor
count      200.00   200.00
mean         2.95  1021.01
std          1.44  1183.20
min          1.00    72.53
25%          2.00   292.07
50%          3.00   592.50
75%          4.00   962.01
max          5.00  4684.02
```

Olhe a coluna `valor`: a **média é 1021** mas a **mediana (50%) é 592**. A média está quase o
dobro da mediana — sinal clássico de que tem venda muito grande puxando o número para cima.

Média sozinha mente. `.describe()` te avisa.

---
# 3. Coluna e filtro

In [ ]:
print(df["valor"].sum().round(2))
print(df["valor"].mean().round(2))
print(df["valor"].max())

`df["valor"]` pega a coluna inteira — e aí `.sum()`, `.mean()`, `.max()`, `.count()` funcionam
direto. **É o acumulador da Aula 4, já pronto.**

E o filtro, que é a operação mais usada de todas:

In [ ]:
caros = df[df["valor"] > 2000]
print(len(caros), "vendas acima de 2000")

Leia devagar, porque a sintaxe assusta:

```
df[ df["valor"] > 2000 ]
 │        └── isso vira uma coluna de True/False, um por linha
 └── e o df fica só com as linhas onde deu True
```

É o `if` da Aula 3 aplicado a 200 linhas de uma vez. O `bool` continua sendo a peça central.

**Duas condições juntas** — e atenção às diferenças:

In [ ]:
santos_monitor = df[(df["cidade"] == "Santos") & (df["produto"] == "monitor")]
print(len(santos_monitor), "monitores vendidos em Santos")

| Em Python puro | No pandas |
|---|---|
| `and` | `&` |
| `or` | `\|` |
| `not` | `~` |

E **cada condição precisa de parênteses**. Sem eles, o Python resolve na ordem errada e você toma
um erro difícil de entender. É chato, é assim, e todo mundo esquece pelo menos uma vez.

---
# 4. `groupby`: a ferramenta que paga a aula

In [ ]:
print(df.groupby("produto")["valor"].agg(["count", "sum", "mean"]).round(2))

```
         count        sum     mean
produto
headset     43   31515.74   732.92
monitor     47  128395.22  2731.81
mouse       34    7687.41   226.10
teclado     47   20629.70   438.93
webcam      29   15973.36   550.81
```

Leia `groupby("produto")` como: **"separa em montinhos por produto, e faz a conta em cada
montinho"**.

Aquela tabela responde três perguntas de negócio de uma vez: quantas vendas, quanto faturou, e o
ticket médio. Monitor é 47 vendas — as mesmas 47 do teclado — mas **seis vezes mais faturamento**.

Na mão, isso seria um dicionário de dicionários e uns quinze `for`.

E dá para agrupar por qualquer coisa, inclusive por um pedaço de texto:

In [ ]:
df["mes"] = df["data"].str[5:7]       # fatiamento da Aula 5, numa coluna inteira
print(df.groupby("mes")["valor"].sum().round(2))

`df["data"].str[5:7]` é o **fatiamento da Aula 5** aplicado a todas as linhas de uma vez.
E `df["mes"] = ...` cria a coluna nova — igual a criar chave nova num dicionário, na Aula 7.

Os 5 maiores:

In [ ]:
print(df.nlargest(5, "valor")[["data", "cidade", "produto", "valor"]])

### 🔧 Exercício

1. Qual **cidade** tem o maior ticket médio?
2. Quantas vendas de `monitor` foram acima de 3000?
3. Some a `quantidade` por cidade e descubra onde saiu mais peça (não mais dinheiro)

In [ ]:
# sua vez

---
# 5. Os três erros que o pandas não avisa

Aqui está o motivo real de esta aula existir. O pandas quase nunca dá erro — **ele te dá um
número errado com cara de certo.**

In [ ]:
import io

sujo = pd.read_csv(io.StringIO('''produto,vendas,nota
teclado,150,4.5
mouse,80,
monitor,900,4.8
headset,250,3.9
webcam,,4.1
'''))

print(sujo)

### 💥 Quebrando de propósito nº 1 — o `NaN` que ninguém vê

`NaN` é como o pandas representa "está faltando". Repare que ele não reclamou de nada ao ler.

In [ ]:
print("média de nota :", sujo["nota"].mean())
print("count de nota :", sujo["nota"].count())
print("len(df)       :", len(sujo))

```
média de nota : 4.325
count de nota : 4       ← conta 4
len(df)       : 5       ← mas são 5 linhas
```

**A média é de quatro notas, não de cinco.** O `.mean()` simplesmente pulou o `NaN` e não avisou.

Isso pode estar certo (a nota faltando não existe mesmo) ou completamente errado (a nota faltando
era zero, e a média real seria 3.46):

In [ ]:
print("ignorando o vazio :", sujo["nota"].mean())
print("tratando como 0   :", sujo["nota"].fillna(0).mean())

> **Você tem que decidir**, e o pandas não vai perguntar. O comando que evita a surpresa:

In [ ]:
print(sujo.isnull().sum())

> Rode `.isnull().sum()` em **toda tabela nova**, antes de qualquer conta. É o `.describe()` do
> dado faltando.

### 💥 Quebrando de propósito nº 2 — a coluna que parece número

In [ ]:
d2 = pd.read_csv(io.StringIO('''produto,valor
teclado,150
mouse,80
monitor,R$ 900
'''))

print(d2["valor"].sum())

```
15080R$ 900
```

**Ele concatenou.** Um `R$` no meio de trezentas linhas fez a coluna inteira virar texto — e a
soma virou aquilo que vocês viram na Aula 2 com `"2" + "2"`.

E ao filtrar:

In [ ]:
print(d2[d2["valor"] > 100])

```
TypeError: '>' not supported between instances of 'str' and 'int'
```

Este pelo menos **grita**. O `.sum()` não gritou — e é aí que mora o perigo.

> `df.dtypes` mostra o tipo de cada coluna. Coluna de número que aparece como texto (`object` ou
> `str`, dependendo da versão) é bandeira vermelha.

### 💥 Quebrando de propósito nº 3 — o filtro que devolve zero

In [ ]:
d3 = pd.DataFrame({"cidade": ["Santos", "Guaruja", "Santos"], "v": [1, 2, 3]})

print(len(d3[d3["cidade"] == "santos"]))    # minúsculo
print(len(d3[d3["cidade"] == "Santos"]))    # como está no arquivo

```
0
2
```

Zero linhas. **Nenhum erro.** É o `"Banana" == "banana"` da Aula 3, agora afetando um relatório
inteiro.

> Quando um filtro voltar vazio, a primeira suspeita é sempre a mesma: caixa alta, espaço nas
> pontas, ou acento. `df["cidade"].unique()` mostra o que existe de verdade na coluna.

---
# 💊 Pílula de IA — 80% do trabalho é limpar tabela

## Etapa 1 — um arquivo como ele chega de verdade

Este CSV são avaliações de clientes com o sentimento marcado. Ele foi montado imitando um
arquivo real: gente diferente digitando, exportação de sistema, planilha passando de mão em mão.

In [ ]:
random.seed(11)

p1 = ["produto excelente", "adorei a qualidade", "atendimento otimo", "material muito bom", "entrega rapida", "compra perfeita"]
p2 = ["chegou antes do prazo", "bem embalado", "recomendo demais", "superou as expectativas", "vale cada centavo", "voltarei a comprar"]
n1 = ["produto pessimo", "qualidade horrivel", "atendimento ruim", "material fraco", "entrega atrasada", "compra frustrante"]
n2 = ["chegou quebrado", "veio errado", "nao recomendo", "propaganda enganosa", "dinheiro jogado fora", "nunca mais compro"]

pos = [f"{a} {b}" for a, b in itertools.product(p1, p2)]
neg = [f"{a} {b}" for a, b in itertools.product(n1, n2)]
random.shuffle(pos)
random.shuffle(neg)

# o teste fica separado desde o começo e NUNCA entra no treino
teste = [(t, "positivo") for t in pos[:12]] + [(t, "negativo") for t in neg[:12]]
pool_pos, pool_neg = pos[12:], neg[12:]

linhas = []
for i in range(160):
    if random.random() < .5:
        t, base = random.choice(pool_pos), "positivo"
    else:
        t, base = random.choice(pool_neg), "negativo"
    r = random.choice([base, base.capitalize(), base.upper(), f" {base} "])
    if random.random() < .12:
        r = ""                       # rótulo faltando
    if random.random() < .10:
        t = t.upper()                # caixa alta
    if random.random() < .08:
        t = "  " + t + "   "         # espaço nas pontas
    linhas.append((t, r))

for _ in range(20):
    linhas.append(random.choice(linhas))   # duplicatas
random.shuffle(linhas)

with open("avaliacoes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["texto", "sentimento"])
    w.writerows(linhas)

print("avaliacoes.csv criado")

In [ ]:
av = pd.read_csv("avaliacoes.csv")

print("linhas:", len(av))
print("rótulos faltando:", int(av["sentimento"].isnull().sum()))
print("linhas duplicadas:", int(av.duplicated().sum()))
print("grafias distintas do rótulo:", int(av["sentimento"].nunique()))
print()
print(av["sentimento"].value_counts(dropna=False))

```
linhas: 180
rótulos faltando: 22
linhas duplicadas: 52
grafias distintas do rótulo: 8

NEGATIVO      29
Negativo      28
POSITIVO      23
NaN           22
 negativo     20
negativo      17
 positivo     14
Positivo      14
positivo      13
```

**Oito grafias para duas coisas.** `NEGATIVO`, `Negativo`, `negativo` e `' negativo '` são a
mesma palavra para você — e quatro rótulos completamente diferentes para o computador. É o
`"Banana" == "banana"` de novo, agora estragando um dataset.

## Etapa 2 — treinando no dado sujo

Vamos só tirar as linhas sem rótulo (não dá para usar) e treinar:

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB


def treinar_e_avaliar(d):
    v = CountVectorizer()
    m = MultinomialNB().fit(v.fit_transform(d["texto"]), d["sentimento"])
    pred = m.predict(v.transform([t[0] for t in teste]))
    certos = sum(1 for p, t in zip(pred, teste) if p == t[1])
    return certos / len(teste) * 100, sorted(d["sentimento"].unique())


sujo_av = av.dropna(subset=["sentimento"])
acerto, classes = treinar_e_avaliar(sujo_av)

print(f"linhas de treino: {len(sujo_av)}")
print(f"classes que o modelo enxerga: {len(classes)}")
print(classes)
print(f"acerto no teste: {acerto:.1f}%")

```
linhas de treino: 158
classes que o modelo enxerga: 8
[' negativo ', ' positivo ', 'NEGATIVO', 'Negativo', 'POSITIVO', 'Positivo', 'negativo', 'positivo']
acerto no teste: 25.0%
```

**25%.** Pior que chutar.

E o motivo está na linha do meio: o modelo achou que existiam **oito sentimentos**. Ele não estava
aprendendo "positivo vs negativo" — estava tentando adivinhar entre oito rótulos, quatro dos quais
significam a mesma coisa.

Nenhum erro. Nenhum aviso. O `fit` rodou lindamente.

## Etapa 3 — quatro linhas de limpeza

In [ ]:
limpo = av.dropna(subset=["sentimento"]).copy()
limpo["texto"] = limpo["texto"].str.strip().str.lower()
limpo["sentimento"] = limpo["sentimento"].str.strip().str.lower()
limpo = limpo.drop_duplicates()

acerto2, classes2 = treinar_e_avaliar(limpo)

print(f"linhas de treino: {len(limpo)}")
print(f"classes: {classes2}")
print(f"acerto no teste: {acerto2:.1f}%")

```
linhas de treino: 46
classes: ['negativo', 'positivo']
acerto no teste: 100.0%
```

De **25% para 100%**.

E olhe o detalhe mais contraintuitivo do curso inteiro: o modelo bom foi treinado com
**46 linhas**, e o ruim com **158**.

**Menos dado, resultado melhor.** Porque as outras 112 linhas eram duplicata ou ruído de rótulo —
elas não eram informação, eram barulho.

> Isso conversa direto com a Aula 8: lá a curva mostrou que mais dado ajuda. Aqui fica a outra
> metade da frase: **mais dado ajuda; mais barulho atrapalha.** E os dois chegam no mesmo arquivo.

## Etapa 4 — agora conte as linhas

| O que | Linhas de código |
|---|---|
| **Limpar** o dado | 4 |
| **Treinar** o modelo | 3 |

Neste exemplo minúsculo, limpeza e treino quase empatam. Num projeto de verdade a limpeza é
carregar de vários lugares, juntar tabelas, resolver data em três formatos, telefone com e sem
DDD, cidade escrita de seis jeitos, valor com vírgula e com ponto, linha duplicada porque o
sistema exportou duas vezes.

**O treino continua sendo três linhas.**

É daí que vem aquela frase que você vai ouvir muito: *80% do trabalho de ciência de dados é
limpeza*. Não é reclamação — é a descrição do trabalho.

---

### O que fica da pílula de hoje

1. **O modelo não sabe que `NEGATIVO` e `negativo` são a mesma coisa.** Você sabe. Por isso a
   limpeza é sua.
2. **Dado sujo não dá erro** — dá resultado ruim com cara de bom. Igual a tudo que a gente viu
   nas pílulas 6, 8 e 9.
3. **Menos dado limpo bate mais dado sujo.** Sempre.

E é por isso que esta é a última pílula separada. Da Aula 11 em diante, o curso inteiro já é
sobre isso.

---
# 🏠 Tarefa da semana

### 1. Relatório de vendas
Usando o `vendas.csv`, responda com pandas e mande as respostas no grupo:

- qual cidade fatura mais, e qual vende mais **peças**
- qual produto tem o maior ticket médio
- qual mês foi o melhor
- as 10 maiores vendas

### 2. Diagnóstico de tabela
Escreva `diagnosticar(df)` que imprime, para qualquer DataFrame: número de linhas, colunas com
valor faltando, quantidade de duplicatas, e o tipo de cada coluna.

É a função que você vai rodar em toda tabela nova pelo resto da vida.

### 3. Limpar de verdade
Pegue o `avaliacoes.csv` e escreva a limpeza **como uma função** `limpar(df)`, com um comentário
explicando cada passo. Depois estrague o arquivo de outro jeito (invente uma sujeira nova) e veja
se a sua função aguenta.

### 4. Desafio
Pegue uma planilha **sua de verdade** — extrato do banco, controle de gastos, planilha do
trabalho — salve como CSV e rode o `diagnosticar()` nela.

Quase sempre tem surpresa. Traga a surpresa no sábado.

---

### 🎓 O fim do Módulo 2

Dez aulas. Vale olhar o que vocês conseguem fazer hoje e não conseguiam em agosto:

- fazer o programa **decidir** (Aula 3) e **repetir** (Aula 4)
- guardar dado em **lista**, **dicionário** e **arquivo** (Aulas 5, 7, 8)
- organizar em **funções** e entregar um **projeto que funciona** (Aula 6)
- **não morrer** no primeiro imprevisto (Aula 9)
- e analisar uma tabela de verdade em três linhas (hoje)

A partir da Aula 11 o curso muda de marcha: `pip`, bibliotecas dos outros, APIs, automação de
verdade — e depois IA de ponta a ponta, sem pílula separada, porque aí já é a aula inteira.

**A pergunta que abre a próxima:** o `pandas` e o `sklearn` não vieram com o Python. Alguém
escreveu, e alguém colocou num lugar de onde a sua máquina baixou.

Quem? Como? E se eu quiser uma dessas para outra coisa — mandar e-mail, mexer em PDF, ler o
WhatsApp?

*(Aula 11: bibliotecas e `pip`.)*